# Baseline 0: TF-IDF and cosine similarity

**Aim:** Freeze our first simple retrieval result before adding more sophisticated methods. This notebook inspects an existing run; it does not tune the query, text, or vectorizer.

TF-IDF means **term frequency–inverse document frequency**. It represents each book using word counts weighted by how common those words are across the books. **Cosine similarity** compares a book's vector with the query vector and gives us a ranking.

| Section | What we establish |
| --- | --- |
| [1. Inputs](#section-1) | Identify the exact query, document file, and one-text-per-book input. |
| [2. Fixed method](#section-2) | See the default settings and fitted vocabulary. |
| [3. Results](#section-3) | Inspect the unchanged top ten and select a book's input text. |
| [4. Validation](#section-4) | Reproduce all ranks and scores and verify unchanged files. |

Create the run once from the repository root with `uv run python -m bookpath.baseline0`. It saves a new folder at `data/evaluation/baseline_0` and protects existing runs from overwrite.

**Quality metrics are not available yet.** nDCG@5, Recall@10, MRR, and Precision@5 need independently judged query–book relevance labels. Cosine similarity supplies the ranking, not those labels. The table in [docs/metrics.md](../docs/metrics.md) records this distinction.

<a id="section-1"></a>

## 1. What exactly did we run?

**Objective:** Load the fixed query and saved results, then show how the document rows become book texts.

We use `data/processed/documents.parquet` from notebook 05. `build_book_corpus()` joins every prepared text belonging to the same `parent_asin`, in saved order, with line breaks. A book's TOC items and chosen Amazon text receive no special weights. `book_title` remains display metadata; it is not prepended to the input text.

In [1]:
import hashlib
import html
import json
from pathlib import Path

import pandas as pd
from IPython.display import HTML, Markdown, display

from bookpath.tfidf import build_book_corpus, rank_books

project_root = Path.cwd()
while not (project_root / "src/bookpath").is_dir():
    if project_root == project_root.parent:
        raise FileNotFoundError("Open this notebook inside the BookPath repository.")
    project_root = project_root.parent

input_path = project_root / "data/processed/documents.parquet"
query_path = project_root / "benchmarks/baseline_0/query.json"
run_path = project_root / "data/evaluation/baseline_0/run.json"
ranking_path = run_path.with_name("ranking.parquet")
if not run_path.is_file():
    raise FileNotFoundError("Run uv run python -m bookpath.baseline0 first.")


def checksum(path):
    with path.open("rb") as source:
        return hashlib.file_digest(source, "sha256").hexdigest()


checksums_before = {
    path: checksum(path) for path in [input_path, query_path, run_path, ranking_path]
}
run = json.loads(run_path.read_text())
query_record = json.loads(query_path.read_text())
documents = pd.read_parquet(input_path)
corpus = build_book_corpus(documents)
ranking = pd.read_parquet(ranking_path)
print("Exact query:", repr(run["query"]))
print(f"{len(documents):,} prepared document rows become {len(corpus):,} book texts.")
print("Method:", run["method"])
print("Run time (UTC):", run["created_at_utc"])
display(corpus[["parent_asin", "book_title", "document_count"]].head(3))
display(Markdown(
    f"**Conclusion:** The candidate set contains **{len(corpus):,} books**, "
    "each represented once. The exact query and prepared text are unchanged."
))

Exact query: 'I want to know what is needed to start a online business, given that i  have a finance background'
150,496 prepared document rows become 9,034 book texts.
Method: baseline_0_tfidf_cosine
Run time (UTC): 2026-10-08T08:03:31.553419+00:00


,parent_asin,book_title,document_count
0,000216132X,"The wheels of commerce, vol.2: civilisation an...",3
1,0007519532,Will there be Donuts?: Better Business One Mee...,6
2,0021057117,California Mathematics Grade 4 (Student Editio...,21


**Conclusion:** The candidate set contains **9,034 books**, each represented once. The exact query and prepared text are unchanged.

<a id="section-2"></a>

## 2. What does the baseline do?

**Objective:** Document one fixed method so later results can be compared with it.

The implementation is `TfidfVectorizer()` followed by `cosine_similarity()`. The vectorizer is fitted on the candidate books only, learning their vocabulary and inverse document frequencies. The query is then transformed with the same fitted vectorizer. The query is not included when learning IDF weights.

The default tokenizer lowercases text and keeps word tokens of at least two characters. The sentence's single-letter `I` and `a` are therefore not tokens. We retain stop words, use single words rather than phrases, and add no stemming, synonym expansion, query rewriting, or parameter search. No settings were changed after inspecting the ranking.

In [2]:
parameter_names = [
    "lowercase", "token_pattern", "ngram_range", "stop_words", "min_df",
    "max_df", "max_features", "use_idf", "smooth_idf", "sublinear_tf", "norm",
]
display(pd.DataFrame([
    {"Setting": name, "Recorded value": repr(run["vectorizer_parameters"][name])}
    for name in parameter_names
]))
print("scikit-learn version:", run["scikit_learn_version"])
print(f"Vocabulary: {run['vocabulary_size']:,} distinct terms.")
print("Query terms found:", run["query_terms_in_vocabulary"])
print("Query terms outside the vocabulary:", run["query_terms_outside_vocabulary"])
display(Markdown(
    "**Conclusion:** Baseline 0 uses the library defaults. "
    f"It represents the candidate books using **{run['vocabulary_size']:,} terms**. "
    "The query uses those same terms and weights; there is no separate finance-background rule."
))

,Setting,Recorded value
0,lowercase,True
1,token_pattern,'(?u)\\b\\w\\w+\\b'
2,ngram_range,"[1, 1]"
3,stop_words,None
4,min_df,1
5,max_df,1.0
6,max_features,None
7,use_idf,True
8,smooth_idf,True
9,sublinear_tf,False


scikit-learn version: 1.9.1
Vocabulary: 56,594 distinct terms.
Query terms found: ['background', 'business', 'finance', 'given', 'have', 'is', 'know', 'needed', 'online', 'start', 'that', 'to', 'want', 'what']
Query terms outside the vocabulary: []


**Conclusion:** Baseline 0 uses the library defaults. It represents the candidate books using **56,594 terms**. The query uses those same terms and weights; there is no separate finance-background rule.

<a id="section-3"></a>

## 3. What does cosine similarity return?

**Objective:** Record the initial top ten exactly as returned, without improving the order.

Rows are sorted by decreasing cosine similarity. Equal scores are ordered by `parent_asin`. A score measures overlap in weighted words; it is not a probability of relevance. One Amazon parent record occupies one rank. Different records for similar titles or editions may appear separately.

In [3]:
with pd.option_context("display.max_colwidth", None, "display.float_format", "{:.6f}".format):
    display(ranking.head(10))

first = ranking.iloc[0]
display(Markdown(
    f"**Conclusion:** The first book is **{first['book_title']}**, with cosine similarity "
    f"**{first['cosine_similarity']:.4f}**. This is the frozen retrieval result, "
    "not a judged recommendation or a quality-metric score. "
    "The repeated Investing Online For Dummies titles remain in their original ranks."
))

,rank,parent_asin,book_title,cosine_similarity
0,1,1683509854,Don’t Sleep on It: Turn Your Passion & Expertise into a Profitable Online Business,0.261789
1,2,1705797547,The 60 Minute Startup: A Proven System to Start Your Business in 1 Hour a Day and Get Your First Paying Customers in 30 Days (or Less) (The Agile Entrepreneurship Series),0.240739
2,3,1119228352,Investing Online For Dummies,0.234915
3,4,0470769262,Investing Online For Dummies,0.232961
4,5,1118495365,Investing Online For Dummies,0.230588
5,6,0793145171,Selling Online: How to Become a Successful E-Commerce Merchant,0.229723
6,7,1581159021,The Pocket Small Business Owner's Guide to Building Your Business (Pocket Small Business Owner's Guides),0.229596
7,8,1118926706,Starting an Online Business All-in-One For Dummies,0.219628
8,9,0764503359,Marketing Online For Dummies,0.205789
9,10,0538496894,Entrepreneurship: Ideas in Action (DECA),0.203651


**Conclusion:** The first book is **Don’t Sleep on It: Turn Your Passion & Expertise into a Profitable Online Business**, with cosine similarity **0.2618**. This is the frozen retrieval result, not a judged recommendation or a quality-metric score. The repeated Investing Online For Dummies titles remain in their original ranks.

**Inspect a book's input:** Leave `example_asin = None` to inspect the first-ranked book, or enter a quoted ID from the table. This changes only the preview, not the baseline ranking. The text below is exactly what TF-IDF receives for that book.

In [4]:
example_asin = None  # Or an existing quoted ID, such as "1119228352".
if example_asin is None:
    example_asin = ranking.iloc[0]["parent_asin"]
selected = corpus.loc[corpus["parent_asin"].eq(example_asin)]
if selected.empty:
    raise ValueError(f"No candidate book has parent_asin = {example_asin!r}.")
book = selected.iloc[0]
print(book["book_title"], "parent_asin =", example_asin)
print(f"Joined documents: {book['document_count']}; input characters: {len(book['text']):,}")
print("Beginning of the input text:")
print(book["text"][:1000])
display(HTML(
    "<details><summary>Read the complete TF-IDF input for this book</summary>"
    f"<div style='max-width: 90ch; white-space: pre-wrap'>{html.escape(book['text'])}</div></details>"
))

Don’t Sleep on It: Turn Your Passion & Expertise into a Profitable Online Business parent_asin = 1683509854
Joined documents: 4; input characters: 962
Beginning of the input text:
Part one. Business fundamentals
Part two. Leveraging your expertise
Part three. Marketing your course.
Don’t Sleep On It!
is a call to action for people to wake up and live their best life by taking what they know and who they are and unleashing it in the world, all while creating their very own profitable online business. So many people want to start a business, but don’t know what to do. With 10 years of online building experience, Kavit Haria guides readers through the steps to turn their passions and interests―their expertise and knowledge―into a profitable online business.
When you light a fire in someone else, you can change the world. Whether you know it or not, what you know at this time is what someone else needs in order to overcome their challenges, remedy their pain, or solve their problem. Stop t

<a id="section-4"></a>

## 4. Can we reproduce the frozen ranking?

**Objective:** Recompute the same method once and compare every book, rank, and score with the saved result. This is a reproducibility check, not a search for better settings.

We also check the recorded input and code checksums, confirm all books appear exactly once, and verify that the four quality metrics remain unmeasured. A failed assertion stops execution before the conclusion.

In [5]:
assert query_record["query_id"] == run["query_id"]
assert query_record["query"] == run["query"]
assert checksums_before[input_path] == run["documents_sha256"]
assert checksums_before[query_path] == run["query_file_sha256"]
assert ranking["parent_asin"].is_unique
assert set(ranking["parent_asin"]) == set(corpus["parent_asin"])
assert ranking["rank"].tolist() == list(range(1, len(corpus) + 1))
assert all(value is None for value in run["metrics"].values())
for name, recorded_checksum in run["code_sha256"].items():
    assert checksum(project_root / "src/bookpath" / name) == recorded_checksum

expected, expected_details = rank_books(corpus, run["query"])
pd.testing.assert_frame_equal(ranking, expected, check_exact=True)
for name, value in expected_details.items():
    assert run[name] == value
for path, recorded_checksum in checksums_before.items():
    assert checksum(path) == recorded_checksum

print(f"Reproduced all {len(ranking):,} book ranks and cosine scores exactly.")
print("Input files, saved run, and ranking remain unchanged.")
display(Markdown(
    "**Conclusion:** Baseline 0 is reproducible. "
    "Its four quality metrics remain unmeasured because no independent relevance labels exist. "
    "This single development query establishes a starting retrieval result; "
    "it does not establish performance across learning goals."
))

Reproduced all 9,034 book ranks and cosine scores exactly.
Input files, saved run, and ranking remain unchanged.


**Conclusion:** Baseline 0 is reproducible. Its four quality metrics remain unmeasured because no independent relevance labels exist. This single development query establishes a starting retrieval result; it does not establish performance across learning goals.

**Next:** Review relevance independently of cosine scores, freeze those labels, and calculate the four metrics. Then compare another retrieval method on the same query and labels. Future methods should not redefine what counts as relevant to match their own rankings.

**Reading:** The supplied *Natural Language Processing in Action* chapter 3 explains TF-IDF and cosine similarity, including scikit-learn's vectorizer example on printed p. 93. The full recorded defaults follow the installed [scikit-learn API](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html).

**Sources:** scikit-learn documents [cosine similarity](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise.cosine_similarity.html). Stanford explains why [evaluation requires independent relevance judgments](https://nlp.stanford.edu/IR-book/html/htmledition/information-retrieval-system-evaluation-1.html). See the complete [project source index](../docs/references.md).